*AI Agents vs Family Guy:*

*Project Background:*

In South Park’s parody, these flashback jokes are shown to be generated by manatees selecting “idea balls” in an aquarium. Each ball contains a random noun, verb, place, or famous person. When five balls are selected, the writers must build the next joke using exactly those items. The joke lands not because it makes sense, but because it barely does — yet still fits the Family Guy formula. To watch that scene:[https://www.youtube.com/watch?v=qTC9j0QpCBM](url)

![](https://www.googleapis.com/download/storage/v1/b/kaggle-user-content/o/inbox%2F1282293%2F2dd422da5036902abe8e79b5bbeaf86d%2FScreenshot%202025-12-03%20at%2018.43.11.png?generation=1764783897115999&alt=media)

This raised a question:
What if we recreate that exact pipeline — the manatee-style “idea ball system” — using a multi-agent AI architecture?

The challenge becomes threefold:

- Can a sequence of agents generate a flashback scenario using the same random-item pipeline mocked in South Park?

- Can another agent, acting as an “investigator,” check whether Family Guy has already created an identical or extremely similar flashback joke?

- And ultimately, can we ever get a “JACKPOT”? //meaning: the generated scenario already exists in a real Family Guy episode.



**We are starting with authentication. //Google Key already added to secrets**

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")


**Then import the neccessary ADK components:**

In [ ]:
from google.adk.agents import Agent, SequentialAgent, LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search #agent ve func sil
from google.genai import types
from google.adk.models import Gemini

print("✅ ADK components imported successfully.")


****To run ADK's web UI later, defining the helper function:****

In [ ]:
# Define helper functions that will be reused throughout the notebook

from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers

# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]['base_url']

    try:
        path_parts = baseURL.split('/')
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix

print("✅ Helper functions defined.")

****To compass some transient errors, to retry several times, if it is needed:****

In [ ]:
#To compass such problems

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)



**THE ARCHITECTURE & THE AGENTS:**



![](https://www.googleapis.com/download/storage/v1/b/kaggle-user-content/o/inbox%2F1282293%2Ffd0d9e79c87d1a45b62a0ce17b246d74%2FScreenshot%202025-12-03%20at%2018.42.56.png?generation=1764783849917795&alt=media)

****FIRST AGENT: CREATING "IDEA BALLS":  "*idea_ball_agent*" : 
//CREATING POSSIBLE IDEA ITEMS SUCH AS VERBS, NOUNS, POP CULTURE REFERENCES AND PLACES****

In the episode, aquarium was full of millions of idea balls, looking like a lottery ball but with some writing such as a verb, a noun, a pop culture or a place. 
//In future this part can be evolved, instead of creating 100 idea balls at the moment, a database can be created with these feature and millions of items,...
For now, in each run, 100 items will be created by the agent randomly to ***{generated_ideaballs}***.

In [ ]:
idea_ball_agent=LlmAgent(
    name="IdeaBallAgent",
    model= Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction= """
You are a word generator.

Generate EXACTLY 100 items. 
For each item, FIRST randomly choose ONE category from A–D, then generate ONE item based on that category:

A. VERB – Any English verb, including new/pop-culture verbs.
B. NOUN – Any noun (object or a person name).
C. POP CULTURE FIGURE – One famous person, using full name or stage name.
D. PLACE – Any place (public, private, fictional, real).

RULES:
- Output EXACTLY 100 items, (*NOT* 100 for each category)
- ONE item per line.
- Each item must be only ONE thing.
- Multi-word names (e.g., "Lady Gaga") count as ONE item.
- NO bullet points, NO numbering, NO explanations.
- DO NOT try to balance categories; let randomness decide.
- DO NOT add any text before or after, only the 100 items.
- DO NOT add the name of the categories as A, B, C or D.

Example format (do NOT copy content):
item1
item2
...
item100
""",
    description="creates a list with 100 items aka 'idea balls' with a verb, noun,pop culture reference or a place.",
    output_key="generated_ideaballs",     # Stores output in state['generated_ideaballs']

)
print("first agent, idea_ball_agent, created")



***SECOND AGENT: PICKING 5 RANDOM IDEA BALLS: "manatee_agent"***

In the episode,the manatees who are living in the big aquarium with millions of idea balls, were picking up some random balls, pushing through pipes, and throughly "selecting" the next Family Guy's episode's scenario items. Here our ****"manatee_agent"**** will pick 5 items from the list **{generated_ideaballs}** created by **idea_ball_agent.**

In [ ]:
manatee_agent=Agent(
    name="ManateeAgent",
    model= Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are items selector.

*From this list strictly" {generated_ideaballs} select *strictly 5 items.
Do *not* add any other text before or after the work.
""",
    description="Picks up 5 items from {generated_ideaballs} created by idea_ball_agent.",
    output_key="picked_ideaballs",    # Stores output in state ['picked_ideaballs']

)

print("second agent, manatee_agent, created")


**THIRD AGENT: SCENARIO WRITER -BASED ON 5 PICKED IDEA BALLS: "scenario_writer_agent"**
In the South Park episode, the “Family Guy” writers’ room is shown creating scenarios based on five idea balls picked by manatees.

Here, our scenario_writer_agent will similarly write two lines of a scenario.

Additional information about Family Guy: throughout the series, there are many scenes where the main character, Peter, flashbacks to his unusual—often impossible—memories. These flashbacks are usually triggered by a transition line such as, “Ah, you think X is bad?”, which jumps the viewer into the past. In these scenes, Peter typically appears somewhere strange, with someone unexpected, doing something absurd. Because these cutaways are always brief, two sentences are enough.

The only rule is that Peter must be the main character in the scenario: in the first sentence, he is somewhere, with someone, doing something; and in the second sentence, they perform another action. The scene is introduced with a typical Family Guy transition line such as, “Ah, you think X is bad?”, which shifts the viewer into the flashback. For that one-shot example is given.

To allow the LLM to produce more unexpected ideas, the temperature is set to 1, so it can generate more unusual and less predictable outputs—just like in those flashbacks.

In [ ]:
scenario_writer_agent=LlmAgent(
    name="ScenarioWriterAgent",
    model= Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    generate_content_config=types.GenerateContentConfig(temperature=1 ,top_p= 0.95), # More non-deterministic, wild output
    instruction="""
    
You are a scenario writer creating short flashbacks.

Use STRICTLY the five items in this list: {picked_ideaballs}

Your output must be EXACTLY two sentences:

1) A setup line where someone asks a person named "Peter" a question:
   "Aa, do you think *[one item from the list]* is bad?"

2) A flashback sentence where Peter remembers something:
   - Peter is the main subject.
   - In the flashback, Peter must somehow involve the remaining 4 items.
   - If one of the items cannot logically be a person / place / action, you may still force it into the memory in any way.

--- ONE-SHOT EXAMPLE (use as style reference) ---
Items given: ["airport", "llama", "wrench", "juice box", "Madonna"]

Sentence 1: "Aa, do you think Madonna is bad?"
Sentence 2: "Peter once found himself at the airport wrestling a llama with a wrench in one hand and a juice box in the other."
--- END OF EXAMPLE ---

Do NOT add anything else. No explanations. No extra text. Only those two sentences.

""",
    description="writes 2 sentences long scenario for the character BLA by using these items only {picked_ideaballs} created by manatee_agent.",
    output_key="generated_scenario",    # Stores output in state ['generated_scenario']

)
print("third agent, scenario_writer_agent, created")

Before continuing, to check how are the agents working and fine tuning, created a sequentialagent with 3 agents. After several changes, at the moment it works fine, there is a possibility that we might find a jackpot* at the end even:) //for jackpot* explanation check the final and the only sequential agent at the end. I'll leave this trial as well to give that example.

In [ ]:
#trial: is it working so far?:: yeap.
""""
deneme6_pipeline_agent = SequentialAgent(
    name="Deneme6PipelineAgent",
    sub_agents=[idea_ball_agent, manatee_agent, scenario_writer_agent],
    description="Executes a sequence of 'creating 100 idea items', picking 5 items out of 100 items, and generating 2 lines of memory scenario with those 5 items.",
    # The agents will run in the order provided: Item generator -> Item picker -> Scenario generator
)

# For ADK tools compatibility, the root agent must be named `root_agent`
root_agent = deneme6_pipeline_agent
"""

In [ ]:
#trial: is it working so far?:: yeap. All agents are working as planned, and output is exciting.
"""runner = InMemoryRunner(agent=deneme6_pipeline_agent)
response = await runner.run_debug(" Give me a scenario please ")"""

**FORTH AGENT: CHECKING THE GENERATED SCENARIO, WHETHER PLAYED ALREADY IN "FAMILY GUY": "inspector_agent"**
With the previous 3 agents, that scene in that episode is  already completed. NOw with this agent we'll check whether there was such a scene already aired in Family Guy series. If completely same: "JACKPOT!", if not agent will try to bring the most similar example to it.
*//In future this can be gamified, and even can be called weekly, just before the next episode will be aired, So there should be waiting moment after scenario agent makes its job, for that episode to start, aired, and once it is finished, from related platform that can be found its english transcription, inspector_agent can inspect, and final result can be given with the root agent. *

In [ ]:
inspector_agent=LlmAgent(
    name="InspectorAgent",
    model= Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
    
You are a professional search assistant/inspector with Google Search capabilities.

Check STRICTLY this {generated_scenario} to see whether it matches any existing short flashback scene of the character "Peter" in a "Family Guy" episode.

Step 1: Use a web search tool to check publicly available transcripts / scripts of Family Guy episodes (e.g. from Drodd.com or Springfield!Springfield!).
Search queries should combine the main elements of the scenario (place, nouns, actions, persons) plus “Family Guy transcript” or “Family Guy script”.

Step 2: If you find an exact match (dialogue, scene description, or situation) from an episode:
   - First output the original {generated_scenario}
   - Then output: “JACKPOT! We have a winner!”
   - Provide the episode name, season and episode number.
   - Provide a short explanation of what happens in that existing scene.

Step 3: If no exact match, but you find an existing scene that shares at least **1 element** (place, noun, action/verb, famous person) with the generated scenario:
   - Output the original {generated_scenario}
   - Then output the closest similar existing example: episode info + brief explanation.

Step 4: Otherwise (0 matching elements):
   - Output the original {generated_scenario}
   - Then output: “Better luck next time, maybe.”

Additional instructions:
- Do NOT add anything else. No introductions, no commentary, no extra text.
- Format your output exactly as described above.

""",
    description="Professional search assistant with Google Search capabilities. Compares the {generated_scenario} created by scenario_writer_agent with the existing Family Guy flashback scene scenarios.",
    tools=[google_search],
    output_key="comparison_result",    # Stores output in state ['comparison_result']

)
print("forth agent, inspector_agent, created")

**The final: root_agent:**

In [ ]:

final_pipeline_agent = SequentialAgent(
    name="FinalPipelineAgent",
    sub_agents=[idea_ball_agent, manatee_agent, scenario_writer_agent,inspector_agent],
    description="Executes a sequence of 'creating 100 idea items', picking 5 items out of 100 items, generating 2 lines of flashback scenario with those 5 items, and compare this short scenario with the existing Family Guy flashback scenes.",
    # The agents will run in the order provided: Item generator -> Item picker -> Scenario generator -> Inspector
)

# For ADK tools compatibility, the root agent must be named `root_agent`
root_agent = final_pipeline_agent


In [ ]:
runner = InMemoryRunner(agent=final_pipeline_agent)
response = await runner.run_debug(" Give me the result please ")

**To check it with ADK WEB UI:**

In [ ]:


#!adk create deneme1_pipeline_agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY 

In [ ]:
#url_prefix = get_adk_proxy_url()

In [ ]:
#!adk web --url_prefix {url_prefix}